# LMS-FNet: Columbia Uncompressed SOTA Evaluation
Domain Adaptation with Data Augmentation for tiny datasets.

In [19]:
import os, gc, io, hashlib, random
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from PIL import Image, ImageChops, ImageEnhance
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
tf.keras.utils.set_random_seed(SEED)
print(f'TensorFlow Version: {tf.__version__}')

TensorFlow Version: 2.20.0


In [20]:
class CrossAttentionFusion(layers.Layer):
    def __init__(self, embed_dim=256, num_heads=4, **kwargs):
        super(CrossAttentionFusion, self).__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

    def build(self, input_shape):
        dim = input_shape[0][-1]
        self.Wq_r = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wq_r')
        self.Wk_e = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wk_e')
        self.Wv_e = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wv_e')
        self.Wq_e = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wq_e')
        self.Wk_r = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wk_r')
        self.Wv_r = self.add_weight(shape=(dim, self.embed_dim), initializer='glorot_uniform', name='Wv_r')
        self.Wo = self.add_weight(shape=(self.embed_dim * 2, self.embed_dim), initializer='glorot_uniform', name='Wo')
        self.bias = self.add_weight(shape=(self.embed_dim,), initializer='zeros', name='bias')

    def _scaled_dot_product(self, Q, K, V):
        scale = tf.math.sqrt(tf.cast(self.head_dim, Q.dtype))
        scores = tf.matmul(Q, K, transpose_b=True) / scale
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, V)

    def call(self, inputs):
        raw_feat, ela_feat = inputs
        Q_r = tf.matmul(raw_feat, self.Wq_r)
        K_e = tf.matmul(ela_feat, self.Wk_e)
        V_e = tf.matmul(ela_feat, self.Wv_e)
        Q_e = tf.matmul(ela_feat, self.Wq_e)
        K_r = tf.matmul(raw_feat, self.Wk_r)
        V_r = tf.matmul(raw_feat, self.Wv_r)
        def reshape_heads(x):
            bs = tf.shape(x)[0]
            x = tf.reshape(x, (bs, self.num_heads, self.head_dim))
            return tf.expand_dims(x, axis=2)
        attn_r2e = self._scaled_dot_product(reshape_heads(Q_r), reshape_heads(K_e), reshape_heads(V_e))
        attn_r2e = tf.reshape(attn_r2e, (-1, self.embed_dim))
        attn_e2r = self._scaled_dot_product(reshape_heads(Q_e), reshape_heads(K_r), reshape_heads(V_r))
        attn_e2r = tf.reshape(attn_e2r, (-1, self.embed_dim))
        combined = tf.concat([attn_r2e, attn_e2r], axis=-1)
        return tf.matmul(combined, self.Wo) + self.bias

    def get_config(self):
        config = super().get_config()
        config.update({'embed_dim': self.embed_dim, 'num_heads': self.num_heads})
        return config

class OHEMFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=3.0, alpha=0.25, hard_ratio=0.20, **kwargs):
        super(OHEMFocalLoss, self).__init__(**kwargs)
        self.gamma, self.alpha, self.hard_ratio = gamma, alpha, hard_ratio
    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred, from_logits=False)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        p_t = tf.clip_by_value(p_t, 1e-7, 1.0 - 1e-7)
        alpha_factor = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        focal_loss = alpha_factor * tf.pow(1.0 - p_t, self.gamma) * bce
        bs = tf.shape(focal_loss)[0]
        k = tf.cast(tf.math.ceil(tf.cast(bs, tf.float32) * self.hard_ratio), tf.int32)
        top_k_loss, _ = tf.math.top_k(focal_loss, k=k)
        return tf.reduce_mean(top_k_loss)
    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'alpha': self.alpha, 'hard_ratio': self.hard_ratio})
        return config


## 2. Dynamic Search for Columbia Dataset

In [21]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
# KEEP THIS AS /kaggle/working ! Do not change it to /kaggle/input !
CACHE_DIR = Path('/kaggle/working/columbia_cache') 

print("Scanning /kaggle/input/ for Columbia '4cam_auth' and '4cam_splc' folders...")
# The script will automatically search through all your inputs to find the images.
input_path = Path('/kaggle/input')
valid_exts = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

au_paths = [str(p) for p in input_path.rglob('*') if '4cam_auth' in p.parts and p.is_file() and p.suffix.lower() in valid_exts]
tp_paths = [str(p) for p in input_path.rglob('*') if '4cam_splc' in p.parts and p.is_file() and p.suffix.lower() in valid_exts]

print(f"Found: {len(au_paths)} Authentic images")
print(f"Found: {len(tp_paths)} Tampered images")

if len(au_paths) == 0 or len(tp_paths) == 0:
    raise ValueError("Columbia dataset not found! Please attach it to this notebook.")

all_paths = au_paths + tp_paths
all_labels = [1]*len(au_paths) + [0]*len(tp_paths)

paths_train, paths_test, y_train, y_test = train_test_split(
    all_paths, all_labels, test_size=0.20, random_state=SEED, stratify=all_labels
)
y_train = np.array(y_train, dtype=np.int32)
y_test = np.array(y_test, dtype=np.int32)

print(f"Training set: {len(paths_train)} images")
print(f"Testing set:  {len(paths_test)} images")


Scanning /kaggle/input/ for Columbia '4cam_auth' and '4cam_splc' folders...
Found: 549 Authentic images
Found: 542 Tampered images
Training set: 872 images
Testing set:  219 images


## 3. Fast Caching and Augmented Dataset Pipeline

In [22]:
def compute_ela(image_path, quality=91, target_size=(224, 224)):
    img = Image.open(image_path).convert('RGB')
    channels = []
    for q in [75, 85, 95]:
        buf = io.BytesIO()
        img.save(buf, 'JPEG', quality=q)
        buf.seek(0)
        compressed = Image.open(buf)
        ela = ImageChops.difference(img, compressed).convert('L')
        extrema = ela.getextrema()
        max_diff = extrema[1] if isinstance(extrema, tuple) else extrema
        if max_diff == 0: max_diff = 1
        ela = ImageEnhance.Brightness(ela).enhance(255.0 / max_diff)
        channels.append(ela)
    mq_ela = Image.merge('RGB', channels)
    return mq_ela.resize(target_size, Image.LANCZOS)

def prepare_cache(paths, split_name, chunk_size=500):
    raw_cache, ela_cache = [], []
    def _proc(p):
        h = hashlib.md5(p.encode()).hexdigest()[:20]
        rp = CACHE_DIR / split_name / 'raw' / f'{h}.jpg'
        ep = CACHE_DIR / split_name / 'ela' / f'{h}.jpg'
        if not rp.exists() or not ep.exists():
            img = Image.open(p).convert('RGB')
            rp.parent.mkdir(parents=True, exist_ok=True)
            img.resize(IMG_SIZE, Image.LANCZOS).save(rp, 'JPEG', quality=85)
            ep.parent.mkdir(parents=True, exist_ok=True)
            compute_ela(p, target_size=IMG_SIZE).save(ep, 'JPEG', quality=90)
        return str(rp), str(ep)
        
    for start in range(0, len(paths), chunk_size):
        chunk = paths[start:start+chunk_size]
        with ThreadPoolExecutor(max_workers=4) as ex:
            futures = [ex.submit(_proc, p) for p in chunk]
            for f in futures:
                rp, ep = f.result()
                raw_cache.append(rp)
                ela_cache.append(ep)
    return raw_cache, ela_cache

print("Generating Columbia Cache...")
raw_train, ela_train = prepare_cache(paths_train, 'train')
raw_test, ela_test = prepare_cache(paths_test, 'test')

def load_image(jpeg_path, png_path):
    raw = tf.image.decode_jpeg(tf.io.read_file(jpeg_path), channels=3)
    ela = tf.image.decode_jpeg(tf.io.read_file(png_path), channels=3)
    raw.set_shape([*IMG_SIZE, 3])
    ela.set_shape([*IMG_SIZE, 3])
    return raw, ela

spatial_augmenter = tf.keras.Sequential([layers.RandomFlip("horizontal_and_vertical")])

def augment_batch(data, label):
    raw, ela = data['raw_input'], data['ela_input']
    combined = tf.concat([raw, ela], axis=-1)
    combined = spatial_augmenter(combined, training=True)
    raw_aug, ela_aug = combined[..., :3], combined[..., 3:]
    raw_aug = tf.image.random_brightness(raw_aug, max_delta=0.1)
    raw_aug = tf.image.random_contrast(raw_aug, lower=0.9, upper=1.1)
    return {'raw_input': raw_aug, 'ela_input': ela_aug}, label

def make_dual_ds(raw_paths, ela_paths, labels, training=False, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices((raw_paths, ela_paths, labels))
    def _process(rp, ep, label):
        raw, ela = load_image(rp, ep)
        raw = tf.keras.applications.densenet.preprocess_input(tf.cast(raw, tf.float32))
        ela = tf.keras.applications.densenet.preprocess_input(tf.cast(ela, tf.float32))
        return {'raw_input': raw, 'ela_input': ela}, label
    
    if training:
        ds = ds.shuffle(1024, seed=seed)
        ds = ds.map(_process, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(BATCH_SIZE, drop_remainder=True)
        ds = ds.map(augment_batch, num_parallel_calls=tf.data.AUTOTUNE)
        return ds.prefetch(1)
    else:
        return ds.map(_process, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(1)

train_ds = make_dual_ds(raw_train, ela_train, y_train, training=True)
test_ds = make_dual_ds(raw_test, ela_test, y_test, training=False)


Generating Columbia Cache...


## 4. Stable Domain Adaptation

In [23]:
strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    model_path = "/kaggle/input/datasets/nikunjkumar05/m1-phase/best_forgery_model.keras"
    if not os.path.exists(model_path):
        model_path = "/kaggle/input/datasets/nikunjkumargond/model1/best_forgery_model.keras"
        
    print(f"Loading Base Model: {model_path}")
    model = tf.keras.models.load_model(
        model_path, 
        custom_objects={'CrossAttentionFusion': CrossAttentionFusion, 'OHEMFocalLoss': OHEMFocalLoss},
        compile=False
    )
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

callbacks = [
    tf.keras.callbacks.ModelCheckpoint('columbia_finetuned.keras', save_best_only=True, monitor='val_auc', mode='max'),
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True)
]

print("Starting Domain Adaptation on Columbia... (15 Epochs)")
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
    callbacks=callbacks
)


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Loading Base Model: /kaggle/input/datasets/nikunjkumar05/m1-phase/best_forgery_model.keras
Starting Domain Adaptation on Columbia... (15 Epochs)
Epoch 1/15
INFO:tensorflow:Collective all_reduce tensors: 682 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


E0000 00:00:1786623999.512216      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/efficientnetb3_ela_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


54/54 ━━━━━━━━━━━━━━━━━━━━ 187s 1s/step - accuracy: 0.5498 - auc: 0.5712 - loss: 0.9015 - val_accuracy: 0.4612 - val_auc: 0.3691 - val_loss: 0.9018
Epoch 2/15
54/54 ━━━━━━━━━━━━━━━━━━━━ 36s 660ms/step - accuracy: 0.6632 - auc: 0.7298 - loss: 0.7034 - val_accuracy: 0.4566 - val_auc: 0.4945 - val_loss: 0.8662
Epoch 3/15
54/54 ━━━━━━━━━━━━━━━━━━━━ 36s 659ms/step - accuracy: 0.7222 - auc: 0.7990 - loss: 0.6194 - val_accuracy: 0.4977 - val_auc: 0.5297 - val_loss: 0.7919
Epoch 4/15
54/54 ━━━━━━━━━━━━━━━━━━━━ 36s 652ms/step - accuracy: 0.7824 - auc: 0.8545 - loss: 0.5479 - val_accuracy: 0.5388 - val_auc: 0.5582 - val_loss: 0.7804
Epoch 5/15
54/54 ━━━━━━━━━━━━━━━━━━━━ 35s 651ms/step - accuracy: 0.8032 - auc: 0.8927 - loss: 0.5029 - val_accuracy: 0.5708 - val_auc: 0.5652 - val_loss: 0.7819
Epoch 6/15
54/54 ━━━━━━━━━━━━━━━━━━━━ 36s 660ms/step - accuracy: 0.8241 - auc: 0.9028 - loss: 0.4813 - val_accuracy: 0.6027 - val_auc: 0.6558 - val_loss: 0.7350
Epoch 7/15
54/54 ━━━━━━━━━━━━━━━━━━━━ 31s 561ms

## 5. Final Evaluation

In [24]:
print("Running Final Inference on Columbia test set...")
test_preds = model.predict(test_ds, verbose=0)
y_pred = (test_preds.ravel() >= 0.5).astype(int)
test_acc = np.mean(y_pred == y_test)
test_auc = roc_auc_score(y_test, test_preds)
cm = confusion_matrix(y_test, y_pred, labels=[0,1])

print("\n" + "="*50)
print("COLUMBIA UNCOMPRESSED RESULTS")
print("="*50)
print(f"Accuracy: {test_acc*100:.2f}%")
print(f"ROC-AUC:  {test_auc:.4f}")
print("Confusion Matrix:")
print(f"True Tampered: {cm[0,0]} | False Authentic: {cm[0,1]}")
print(f"False Tampered: {cm[1,0]} | True Authentic: {cm[1,1]}")
print("="*50)


Running Final Inference on Columbia test set...

COLUMBIA UNCOMPRESSED RESULTS
Accuracy: 90.41%
ROC-AUC:  0.9814
Confusion Matrix:
True Tampered: 98 | False Authentic: 11
False Tampered: 10 | True Authentic: 100
